## Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_poisson_deviance
from datasets import Dataset
import statsmodels.api as sm
import matplotlib.pyplot as plt


## Raw Data GLM

In [2]:
# ==========================================
# DATA LOADING & PREPROCESSING
# ==========================================
print("Loading freMTPL2freq dataset...")
dataset = fetch_openml(data_id=41214, as_frame=True)
dat = dataset.frame

# Clean basic types first
dat['ClaimNb'] = pd.to_numeric(dat['ClaimNb'])
dat['Exposure'] = pd.to_numeric(dat['Exposure'])
dat['Exposure'] = dat['Exposure'].clip(upper=1.0)
dat['Frequency'] = dat['ClaimNb'] / dat['Exposure']

# Perform same mapping and engineering as Statistical Foundations for Actuarial Learning 5.2.4.
dat = pd.get_dummies(dat, columns=['VehGas'],drop_first=True)
dat = pd.get_dummies(dat, columns=['VehBrand'],drop_first=True)
dat = pd.get_dummies(dat, columns=['Region'],drop_first=True)
area_remapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6
}
dat["Area"] = dat["Area"].map(area_remapping)

dat['VehPowerGLM'] = pd.Categorical(np.minimum(dat['VehPower'], 9))

dat['VehAgeGLM'] = pd.cut(
    dat['VehAge'],
    bins=[0, 5, 12, 101],
    labels=["0-5", "6-12", "12+"],
    include_lowest=True
)

dat['DrivAgeGLM'] = pd.cut(
    dat['DrivAge'],
    bins=[18, 20, 25, 30, 40, 50, 70, 101],
    labels=["18-20", "21-25", "26-30", "31-40", "41-50", "51-70", "71+"],
    include_lowest=True
)

dat['BonusMalusGLM'] = np.minimum(dat['BonusMalus'], 150)

dat['DensityGLM'] = np.log(dat["Density"].astype(float))

dat = pd.get_dummies(dat, columns=['DrivAgeGLM'],drop_first=True)
dat = pd.get_dummies(dat, columns=['VehAgeGLM'],drop_first=True)
dat = pd.get_dummies(dat, columns=['VehPowerGLM'],drop_first=True)

dat = dat.drop(["VehPower", "VehAge", "DrivAge", "BonusMalus","Density"], axis = 1)
dat = dat.astype('float32')

Loading freMTPL2freq dataset...


In [3]:
# Load the split indices
df_splits = pd.read_csv('../freMTPL2freq_split_indices.csv')

# Ensure IDpol is the same type in both dataframes for a clean merge
dat['IDpol'] = dat['IDpol'].astype(int)
df_splits['IDpol'] = df_splits['IDpol'].astype(int)

# Merge the dataset with the split indicators
# We use a left join to keep the original data rows
df_merged = dat.merge(df_splits, on='IDpol', how='left')

# Create the subsets based on the indicator columns
train_df = df_merged[df_merged['is_train'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
test_df = df_merged[df_merged['is_test'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
finetune_df = df_merged[df_merged['is_finetune'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()

# Print results
print(f"Total rows: {len(dat)}")
print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Finetune rows: {len(finetune_df)}")

# Inspect the training set
print(train_df.head())

Total rows: 678013
Train rows: 500000
Test rows: 100000
Finetune rows: 76782
   IDpol  ClaimNb  Exposure  Area  Frequency  VehGas_'Regular'  VehBrand_B10  \
1      3      1.0      0.77   4.0   1.298701               1.0           0.0   
3     10      1.0      0.09   2.0  11.111111               0.0           0.0   
4     11      1.0      0.84   2.0   1.190476               0.0           0.0   
5     13      1.0      0.52   5.0   1.923077               1.0           0.0   
7     17      1.0      0.27   3.0   3.703704               0.0           0.0   

   VehBrand_B11  VehBrand_B12  VehBrand_B13  ...  DrivAgeGLM_41-50  \
1           0.0           1.0           0.0  ...               0.0   
3           0.0           1.0           0.0  ...               1.0   
4           0.0           1.0           0.0  ...               1.0   
5           0.0           1.0           0.0  ...               0.0   
7           0.0           1.0           0.0  ...               0.0   

   DrivAgeGLM_51-70  

In [4]:
train_df = train_df.drop(["Frequency", "IDpol"], axis = 1).astype('float32')
test_df = test_df.drop(["Frequency", "IDpol"], axis = 1).astype('float32')

In [5]:
# Separate Predictors from Meta Data
meta_cols = ['ClaimNb', 'Exposure']
pred_cols = train_df.columns.difference(meta_cols)

X_train = train_df[pred_cols]
meta_train = train_df[meta_cols]

X_test = test_df[pred_cols]
meta_test = test_df[meta_cols]

# Concatenate 
final_train = pd.concat([X_train, meta_train], axis=1)
final_test = pd.concat([X_test, meta_test], axis=1)

print(final_train.head())

   Area  BonusMalusGLM  DensityGLM  DrivAgeGLM_21-25  DrivAgeGLM_26-30  \
1   4.0           50.0    7.104144               0.0               0.0   
3   2.0           50.0    4.330733               0.0               0.0   
4   2.0           50.0    4.330733               0.0               0.0   
5   5.0           50.0    8.007367               0.0               0.0   
7   3.0           68.0    4.919981               0.0               0.0   

   DrivAgeGLM_31-40  DrivAgeGLM_41-50  DrivAgeGLM_51-70  DrivAgeGLM_71+  \
1               0.0               0.0               1.0             0.0   
3               0.0               1.0               0.0             0.0   
4               0.0               1.0               0.0             0.0   
5               1.0               0.0               0.0             0.0   
7               1.0               0.0               0.0             0.0   

   Region_R21  ...  VehBrand_B5  VehBrand_B6  VehGas_'Regular'  VehPowerGLM_5  \
1         0.0  ...     

In [6]:
# ---------------------------------------------------------
# Prepare Training Data
# ---------------------------------------------------------
# Identify predictor columns (The PCs)
# We exclude 'response', 'offset', and vehicle age one hot encoded variables to get just the X matrix
predictors = [
    c for c in final_train.columns 
    if c not in ['ClaimNb', 'Exposure'] and not c.startswith('VehAgeGLM')
]

X_train = final_train[predictors]
y_train = final_train['ClaimNb']

# Statsmodels does not add an intercept by default.
# PCA centers data, but you still need an intercept for the baseline rate.
X_train = sm.add_constant(X_train)

# Define the offset for training
train_offset = np.log(final_train['Exposure'])

# ---------------------------------------------------------
# Fit the Poisson GLM
# ---------------------------------------------------------
# Instantiate the model
glm_model = sm.GLM(
    endog=y_train, 
    exog=X_train, 
    offset=train_offset,
    family=sm.families.Poisson()
)

# Fit the model
results = glm_model.fit()

print(results.summary())

# ---------------------------------------------------------
# Predict on Test Data
# ---------------------------------------------------------
X_test = final_test[predictors]
X_test = sm.add_constant(X_test, has_constant='add') # Ensure structure matches Train

# Define offset for testing
test_offset = np.log(final_test['Exposure'])

# Generate Predictions (Expected Counts)
# We pass the test offset here so predictions are scaled correctly
predictions = results.predict(exog=X_test, offset=test_offset)

# Add predictions back to the test dataframe for analysis
final_test['predicted_counts'] = predictions

print("\nFirst 5 Predictions:")
print(final_test[['ClaimNb', 'Exposure', 'predicted_counts']].head())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               500000
Model:                            GLM   Df Residuals:                   499953
Model Family:                 Poisson   Df Model:                           46
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0496e+05
Date:                Mon, 04 May 2026   Deviance:                   1.5913e+05
Time:                        17:11:22   Pearson chi2:                 1.30e+06
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01012
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -4.0102      0.075  

In [7]:
# Access the Poisson family object from your fitted model
poisson_family = results.family

# Calculate Total Deviance for the Test Set
# Inputs: (Endog/Actual, Mu/Predicted)
test_deviance_total = poisson_family.deviance(
    final_test['ClaimNb'], 
    final_test['predicted_counts']
)

# Calculate Mean Deviance
test_mean_deviance = test_deviance_total / len(final_test)

print(f"Test Mean Poisson Deviance: {test_mean_deviance:.4f}")

Test Mean Poisson Deviance: 0.3219
